# 12. Keep the inputs an experiment used

## Objective

Save a small study, change the original data, and show that the saved study still uses its original values.

We use two made-up products over nine days: 18 rows, with no products excluded. Days 1 to 7 are training data. Days 8 and 9 are test data. There is no validation split in this lesson.

A hash is a fingerprint used to detect changes to saved contents. Start with `hash`, which needs no database. The optional `table` and `table+hash` settings use an already configured protected PostgreSQL archive. Follow notebook 10 first if training and test DataFrames are new to you.

A saved capture is a separate copy of the inputs. It is not a live table name plus a timestamp. Database history cannot recover an upload that was never captured after that history has expired.


## Step 1: choose how to save the inputs

Leave `mode` as `hash` for the first run. It saves files and checks their recorded hashes.

Choose `table` to save protected database captures without hashing every input row. Choose `table+hash` to use those captures and also check their row hashes.

For either table setting, first run `python -m pip install '.[postgres]'` from the checkout in the activated notebook environment. The administrator's setup environment needs the driver too. Then follow the [protected archive setup](README.md#set-up-a-protected-postgresql-archive) with your database administrator. The archive owner, registration, reader and release roles must be separate. Put connection strings in the environment before starting Jupyter, never in this notebook. Set `AF_TUTORIAL_SNAPSHOT_SCHEMA` to that archive's schema. The code below names the environment variables; it does not contain credentials.

A registry records saved studies, their IDs and where their inputs live. Each run of this notebook uses a new folder so it cannot overwrite an earlier lesson.


In [ ]:
import os
import uuid
from pathlib import Path

from autoforecast.study_registration import StudyRegistry
from pyspark.sql import SparkSession

# Use files first. Change this value only after setting up a protected archive.
mode = "hash"  # "hash", "table", or "table+hash"

# Give this lesson its own output folder outside the library checkout.
base = Path(os.environ.get("AUTOFORECAST_TUTORIAL_DIR", "~/autoforecast-tutorials")).expanduser()
work = base / ("saved-inputs-" + uuid.uuid4().hex)
work.mkdir(parents=True)

# Start a small local Spark session to prepare our example data.
spark = (SparkSession.builder.master("local[2]")
         .config("spark.sql.shuffle.partitions", "2")
         .config("spark.ui.enabled", "false").getOrCreate())

# Table settings use named credentials and an explicitly selected destination.
options = {}
if mode != "hash":
    options = {
        "connection_settings": {"archive": {
            "kind": "postgresql",
            "dsn_env": "AF_TUTORIAL_READER_DSN",
            "writer_dsn_env": "AF_TUTORIAL_WRITER_DSN",
            "release_dsn_env": "AF_TUTORIAL_RELEASE_DSN",
        }},
        "snapshots": {
            "connection": "archive",
            "namespace": os.environ["AF_TUTORIAL_SNAPSHOT_SCHEMA"],
        },
    }
registry = StudyRegistry(work / "registry", spark=spark, snapshot_mode=mode, **options)
print("Chosen setting:", mode)
print("Lesson folder:", work)


### Understand the output

The output shows the chosen setting and a new lesson folder. This cell stores your choices; it has not checked the archive yet. Registration in Step 3 checks the required protection. If it cannot establish that protection, it stops instead of switching storage modes or using the live source.


## Step 2: make data that is easy to check

Each product has the same values: 1 on the first day, 2 on the second day, and so on. We keep the test dates out of training.


In [ ]:
from datetime import date

from pyspark.sql import functions as F

# Make all 18 observations and write a source file for this lesson.
original_rows = [(product, date(2024, 1, day), float(day))
                 for product in ("A", "B") for day in range(1, 10)]
source_path = work / "source.parquet"
spark.createDataFrame(original_rows, "product string, day date, units double").write.parquet(
    str(source_path)
)

# Read the saved source, then split it using our declared dates.
source = spark.read.parquet(str(source_path))
training = source.filter(F.col("day") <= date(2024, 1, 7))
testing = source.filter(F.col("day") >= date(2024, 1, 8))
print("Source rows:", source.count())
print("Training rows:", training.count())
print("Test rows:", testing.count())


### Understand the output

Expect 18 source rows, 14 training rows and 4 test rows. Both products are included. These counts describe the population; matching counts alone would not prove that the values stayed unchanged.


## Step 3: register the study

Registration saves the inputs and the evaluation rules. Keep the returned study ID. Using that ID later reopens the saved inputs instead of reading new source values.


In [ ]:
from autoforecast import register_study

# Save the two stages and the rules for comparing forecasts with actual values.
study_id = register_study(
    registry=registry,
    snapshot_mode=mode,
    training_data=training,
    testing_data=testing,
    series_id=["product"], date_col="day", target_col="units",
    target_unit="units", frequency="D", metric="mae",
    training_date_range=("2024-01-01", "2024-01-07"),
    testing_date_range=("2024-01-08", "2024-01-09"),
    feature_roles={"season_length": 1},
)
print("Saved study:", study_id)


### Understand the output

The ID identifies this saved study. All three settings return the same kind of ID and use the same model-running API. Separate table captures can have different IDs even if they contain equal rows.


## Step 4: change the source and reopen the saved inputs

Imagine a later upload replaces every value with 999. We overwrite the lesson's source file with those new values. Then we compare every saved product, date and value with the original rows. Refit data is the history used to fit the final model before testing. With no validation stage here, it matches the training data. Training and refitting each use days 1 to 7; testing uses days 8 and 9. There is no validation stage.

Collecting rows into Python is suitable for these 18 toy observations. Do not collect a large dataset this way.


In [ ]:
# Overwrite the source using new rows, without reading the file being replaced.
changed_rows = [(product, day, 999.0) for product, day, units in original_rows]
spark.createDataFrame(changed_rows, "product string, day date, units double").write.mode(
    "overwrite"
).parquet(str(source_path))
current_rows = [tuple(row) for row in spark.read.parquet(str(source_path)).collect()]
assert sorted(current_rows) == sorted(changed_rows)
print("Current source values:", sorted({row[2] for row in current_rows}))

# Compare every saved row, including repeated rows, with the original toy data.
expected_training = sorted(row for row in original_rows if row[1] <= date(2024, 1, 7))
expected_testing = sorted(row for row in original_rows if row[1] >= date(2024, 1, 8))
with registry.load_study(study_id) as study:
    for stage, expected in [("training", expected_training), ("refit", expected_training),
                            ("testing", expected_testing)]:
        frame = getattr(study, stage + "_data")
        actual = sorted(tuple(row) for row in frame.select("product", "day", "units").collect())
        assert actual == expected, stage + " rows differ from the original data"
        print(stage, "original rows checked:", len(actual))
    assert study.validation_data is None


### Understand the output

The current source contains only 999. The saved study still has all 14 original training rows, the same 14 refit rows, and all 4 original test rows. These are overlapping views of 18 observations, not 32 separate source observations. Each assertion checks full rows, not just their count.

In table mode, ordinary source, registration and training accounts must also be unable to edit the saved capture itself. PostgreSQL enforces that through the protected archive. An administrator can bypass ordinary permissions, so administrators and storage operators remain trusted. `table+hash` adds a content check; it cannot restore deleted data.


## Step 5: run a simple forecast on the saved study

With a season length of one, the baseline repeats the last training value, 7. The actual values are 8 and 9. The average absolute error is therefore `(1 + 2) / 2 = 1.5` for each product.


In [ ]:
from autoforecast import run_study

# Forecast from the original study ID; do not register the changed source again.
result = run_study(study_id, "seasonal_naive", registry=registry, publish=False)
print("Test MAE:", result.testing_metrics["mae"])
assert result.testing_metrics["mae"] == 1.5


### Understand the output

Expect an MAE of 1.5 for this baseline. That score illustrates the forecast calculation; the full row comparisons in Step 4 check the saved data. This is not a speed benchmark or evidence that this model is suitable for your own data.

`publish=False` means do not publish result tables. It does not stop registration from saving the experiment inputs.


## Step 6: close this lesson

Close readers before releasing a study. Table captures remain in the archive until an explicit release. Releasing them requires the separate release account and makes this study unusable for future runs. Keep them if you want to reopen the lesson later.

This cell closes connections and Spark. It leaves the saved study and results in the printed lesson folder.


In [ ]:
# Close only resources owned by this registry and notebook.
registry.close()
spark.stop()
print("Saved lesson:", work)


### Understand the output

The saved files remain outside the source checkout. For a table study, its protected captures also remain in the configured database. Use `registry.release(study_id)` with the separate release credentials when the study is no longer needed; do not delete its tables manually or change completion records.
